In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
print("Plotly version:", __import__("plotly").__version__)

Plotly version: 5.24.1


In [2]:
!pip install plotly pandas

In [5]:
uploaded = files.upload()
file_name = next(iter(uploaded))

Saving University_Student_Performance_10000_Sample_20pct.csv to University_Student_Performance_10000_Sample_20pct.csv


In [6]:
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Available columns:", df.columns.tolist())
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())

Dataset loaded successfully!
Available columns: ['Student_ID', 'Age', 'Gender', 'Department', 'Year', 'Attendance_%', 'Study_Hours_Day', 'Assignment_Score', 'Internal_Marks', 'Final_Marks', 'Total_Score', 'Grade', 'Result']
Rows: 1990
Columns: 13


,Student_ID,Age,Gender,Department,Year,Attendance_%,Study_Hours_Day,Assignment_Score,Internal_Marks,Final_Marks,Total_Score,Grade,Result
0,12556,19.0,Female,CSE,3rd Year,83.005542,0.452553,75.604479,46.856789,45.123093,53.350233,D,Pass
1,18187,18.0,Male,EEE,1st Year,59.562456,3.051305,79.679935,64.203428,63.794833,67.909117,C,Pass
2,16387,21.0,Female,CSE,4th Year,75.602251,1.965060,98.934149,65.437051,53.060774,68.860815,C,Pass
3,11094,18.0,Female,ECE,4th Year,77.171696,3.888909,59.915450,67.359953,49.185465,58.229032,D,Pass
4,11703,18.0,Male,ECE,3rd Year,82.500646,4.713640,56.019531,77.524515,68.182205,68.411345,C,Pass


In [7]:
fig1 = px.histogram(
    df,
    x='Total_Score',
    nbins=20,
    title='Graph 1: Distribution of Student Total Scores',
    labels={'Total_Score': 'Total Score', 'count': 'Number of Students'},
    hover_data=['Grade', 'Result']
)

fig1.update_layout(
    xaxis_rangeslider_visible=True,
    template='plotly_white'
)
fig1.show()

In [8]:
fig2 = px.scatter(
    df,
    x='Study_Hours_Day',
    y='Total_Score',
    color='Grade',
    hover_data=['Student_ID', 'Department', 'Attendance_%', 'Result'],
    title='Graph 2: Study Hours per Day vs Total Score',
    labels={
        'Study_Hours_Day': 'Study Hours per Day',
        'Total_Score': 'Total Score',
        'Grade': 'Grade'
    }
)

fig2.update_layout(template='plotly_white')
fig2.show()

In [9]:
year_order = ['1st Year', '2nd Year', '3rd Year', '4th Year']

fig3 = px.box(
    df,
    x='Year',
    y='Total_Score',
    color='Year',
    category_orders={'Year': year_order},
    points='all',
    hover_data=['Student_ID', 'Department', 'Grade', 'Result'],
    title='Graph 3: Student Total Score Distribution by Academic Year',
    labels={
        'Year': 'Academic Year',
        'Total_Score': 'Total Score'
    }
)

fig3.update_layout(template='plotly_white')
fig3.show()

In [10]:
fig4 = px.scatter_3d(
    df,
    x='Study_Hours_Day',
    y='Attendance_%',
    z='Total_Score',
    color='Grade',
    hover_data=['Student_ID', 'Department', 'Year', 'Result'],
    title='Graph 4: 3D Analysis of Student Performance',
    labels={
        'Study_Hours_Day': 'Study Hours / Day',
        'Attendance_%': 'Attendance (%)',
        'Total_Score': 'Total Score'
    }
)

fig4.update_layout(template='plotly_white')
fig4.show()

In [11]:
animation_df = df.dropna(subset=['Year']).copy()
# Sort by Year order for chronological animation frames
year_order = ['1st Year', '2nd Year', '3rd Year', '4th Year']
animation_df['Year'] = pd.Categorical(animation_df['Year'], categories=year_order, ordered=True)
animation_df = animation_df.sort_values('Year')

fig5 = px.scatter(
    animation_df,
    x='Study_Hours_Day',
    y='Total_Score',
    animation_frame='Year',
    size='Assignment_Score',
    color='Grade',
    hover_data=['Student_ID', 'Department', 'Attendance_%', 'Result'],
    range_x=[
        animation_df['Study_Hours_Day'].min() - 0.5,
        animation_df['Study_Hours_Day'].max() + 0.5
    ],
    range_y=[
        animation_df['Total_Score'].min() - 5,
        animation_df['Total_Score'].max() + 5
    ],
    size_max=15,
    title='Graph 5: Student Academic Trends by Year',
    labels={
        'Study_Hours_Day': 'Study Hours per Day',
        'Total_Score': 'Total Score',
        'Assignment_Score': 'Assignment Score',
        'Year': 'Academic Year'
    }
)

fig5.update_layout(template='plotly_white')
fig5.show()

In [12]:
dropdown_df = df.dropna(subset=['Total_Score']).copy()
metrics = ['Total_Score', 'Assignment_Score', 'Internal_Marks']

fig6 = go.Figure()
visibility = []

for i, metric in enumerate(metrics):
    fig6.add_trace(
        go.Scatter(
            x=dropdown_df['Study_Hours_Day'],
            y=dropdown_df[metric],
            mode='markers',
            name=metric.replace('_', ' '),
            visible=(i == 0),
            text=dropdown_df['Student_ID'],
            customdata=dropdown_df[['Department', 'Grade']].values,
            hovertemplate=(
                'Study Hours: %{x}<br>'
                + metric.replace('_', ' ') + ': %{y}<br>'
                + 'Student ID: %{text}<br>'
                + 'Department: %{customdata[0]}<br>'
                + 'Grade: %{customdata[1]}'
                + '<extra></extra>'
            )
        )
    )

for i, metric in enumerate(metrics):
    visible = [j == i for j in range(len(metrics))]
    visibility.append(visible)

buttons = []
for i, metric in enumerate(metrics):
    buttons.append(
        dict(
            label=metric.replace('_', ' ').upper(),
            method='update',
            args=[
                {'visible': visibility[i]},
                {'yaxis': {'title': metric.replace('_', ' ').upper()}}
            ]
        )
    )

fig6.update_layout(
    title='Graph 6: Analyze Academic Scores Using the Dropdown Menu',
    xaxis_title='Study Hours per Day',
    yaxis_title=metrics[0].replace('_', ' ').upper(),
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=1.15,
        y=1.15
    )],
    template='plotly_white'
)

fig6.show()

In [13]:
figures = [fig1, fig2, fig3, fig4, fig5, fig6]
names = [
    '01_histogram',
    '02_scatter',
    '03_boxplot',
    '04_3d_scatter',
    '05_animated_scatter',
    '06_dropdown_scatter'
]

for fig, name in zip(figures, names):
    fig.write_html(name + '.html')

html_parts = []
for i, fig in enumerate(figures):
    html_parts.append(
        pio.to_html(
            fig,
            full_html=False,
            include_plotlyjs='cdn' if i == 0 else False
        )
    )

report = """
<html>
<head>
<meta charset="utf-8">
<title>Lab Assignment 6 - Student Performance Visualization</title>
</head>
<body>
<h1>Lab Assignment 6: Interactive Student Performance Visualization</h1>
<p>Data Analytics and Visualization</p>
""" + "\n".join(html_parts) + "</body></html>"

with open('Lab_Assignment_6_Report.html', 'w', encoding='utf-8') as f:
    f.write(report)

print("All six graphs created successfully!")

All six graphs created successfully!
